## 0. Введение

Этот ноутбук демонстрирует базовый запуск обучения распознавателя `TRBA` в версии `0.1.12` и выше. В примере для быстрого теста используется валидационная часть набора данных как обучающая и валидационная выборка, поэтому он предназначен прежде всего для демонстрации процесса запуска обучения.

Пример был апробирован в среде Google Colab в конфигурации с 12 ГБ оперативной памяти и видеокартой NVIDIA T4.

Минимальные технические требования для запуска примера:

- Python 3.8+
- не менее 12 ГБ оперативной памяти
- NVIDIA GPU с объемом видеопамяти не менее 8 ГБ

Следует учитывать, что данный пример относится к обучению модели распознавания, поэтому запуск на GPU является рекомендуемым сценарием. Запуск на CPU возможен, но будет существенно медленнее и не рассматривается как базовый вариант для апробации.

## 1. Установка зависимостей

In [ ]:
#может потребовать перезапуска среды в Google Colab
!pip install "manuscript-ocr[dev]>=0.1.12"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.2/574.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.3/69.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB

In [ ]:
# Затем обновите PyTorch (если требуется) на GPU версию, совместимую с вашей версией CUDA (например, для CUDA 11.8):
pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu118

In [1]:
!pip install rarfile

## 2. Скачивание данных

In [2]:
from pathlib import Path
from urllib.request import urlretrieve

import rarfile

url = "https://huggingface.co/datasets/sherstpasha/YeniseiGovReports-HWR/resolve/main/YeniseiGovReports-HWR.rar"

work_dir = Path("datasets")
archive_path = work_dir / "YeniseiGovReports-HWR.rar"
extract_dir = work_dir / "YeniseiGovReports-HWR"

work_dir.mkdir(parents=True, exist_ok=True)

if not archive_path.exists():
    urlretrieve(url, archive_path)

if not any(extract_dir.rglob("labels.csv")):
    extract_dir.mkdir(parents=True, exist_ok=True)
    try:
        with rarfile.RarFile(archive_path) as rf:
            rf.extractall(extract_dir)
    except rarfile.RarCannotExec as e:
        raise RuntimeError(
            "rarfile установлен, но в системе не найден архиватор для распаковки RAR. "
            "Установите один из архиваторов: unrar, unar, 7z или 7zz."
        ) from e

def find_file(split: str, name: str):
    return next((str(p) for p in extract_dir.rglob(name) if p.parent.name == split), None)

def find_dir(split: str, name: str):
    return next((str(p) for p in extract_dir.rglob(name) if p.is_dir() and p.parent.name == split), None)

train_csvs = find_file("train", "labels.csv")
train_roots = find_dir("train", "img")
val_csvs = find_file("val", "labels.csv")
val_roots = find_dir("val", "img")
test_csvs = find_file("test", "labels.csv")
test_roots = find_dir("test", "img")

print("archive_path =", archive_path)
print("extract_dir  =", extract_dir)
print("train_csvs   =", train_csvs)
print("train_roots  =", train_roots)
print("val_csvs     =", val_csvs)
print("val_roots    =", val_roots)
print("test_csvs    =", test_csvs)
print("test_roots   =", test_roots)


archive_path = datasets/YeniseiGovReports-HWR.rar
extract_dir  = datasets/YeniseiGovReports-HWR
train_csvs   = datasets/YeniseiGovReports-HWR/YeniseiGovReports-HWR/train/labels.csv
train_roots  = datasets/YeniseiGovReports-HWR/YeniseiGovReports-HWR/train/img
val_csvs     = datasets/YeniseiGovReports-HWR/YeniseiGovReports-HWR/val/labels.csv
val_roots    = datasets/YeniseiGovReports-HWR/YeniseiGovReports-HWR/val/img
test_csvs    = None
test_roots   = None


## 3. Запуск обучения

In [3]:
from pathlib import Path

from manuscript.recognizers import TRBA

dataset_root = Path(extract_dir) / "YeniseiGovReports-HWR" / "val"

val_csvs = str(dataset_root / "labels.csv")
val_roots = str(dataset_root / "img")

# Для быстрого теста используем валидационный набор и как train, и как val
train_csvs = val_csvs
train_roots = val_roots

print("train_csvs  =", train_csvs)
print("train_roots =", train_roots)
print("val_csvs    =", val_csvs)
print("val_roots   =", val_roots)

TRBA.train(
    train_csvs=train_csvs,
    val_csvs=val_csvs,
    train_roots=train_roots,
    val_roots=val_roots,
    cnn_backbone="seresnet31lite",
    epochs=1,
)

[2026-04-10 10:52:49] Start training
INFO:train:Start training
[2026-04-10 10:52:49] Experiment dir: exp1
INFO:train:Experiment dir: exp1
[2026-04-10 10:52:49] Seed: 42
INFO:train:Seed: 42
[2026-04-10 10:52:49] Saved config to exp_dir/config.json
INFO:train:Saved config to exp_dir/config.json
[2026-04-10 10:52:49] Device: cuda
INFO:train:Device: cuda
[2026-04-10 10:52:49] Charset loaded: 194 tokens
INFO:train:Charset loaded: 194 tokens
[2026-04-10 10:52:49] Charset copied to experiment dir: exp1/charset.txt
INFO:train:Charset copied to experiment dir: exp1/charset.txt


train_csvs  = datasets/YeniseiGovReports-HWR/YeniseiGovReports-HWR/val/labels.csv
train_roots = datasets/YeniseiGovReports-HWR/YeniseiGovReports-HWR/val/img
val_csvs    = datasets/YeniseiGovReports-HWR/YeniseiGovReports-HWR/val/labels.csv
val_roots   = datasets/YeniseiGovReports-HWR/YeniseiGovReports-HWR/val/img


[2026-04-10 10:52:49] Using default pretrained weights: trba_lite_g1.pth (GitHub release)
INFO:train:Using default pretrained weights: trba_lite_g1.pth (GitHub release)
[2026-04-10 10:52:49] Default pretrain config: https://github.com/konstantinkozhin/manuscript-ocr/releases/download/v0.1.0/trba_lite_g1.json
INFO:train:Default pretrain config: https://github.com/konstantinkozhin/manuscript-ocr/releases/download/v0.1.0/trba_lite_g1.json


Downloading: "https://github.com/konstantinkozhin/manuscript-ocr/releases/download/v0.1.0/trba_lite_g1.pth" to /root/.cache/torch/hub/checkpoints/trba_lite_g1.pth


100%|██████████| 36.2M/36.2M [00:01<00:00, 24.5MB/s]
[2026-04-10 10:52:51] Pretrain load summary: loaded=245/245 keys; missing=0; shape_mismatch=0
INFO:train:Pretrain load summary: loaded=245/245 keys; missing=0; shape_mismatch=0
[2026-04-10 10:52:51] Freeze policy applied: cnn: NONE (no freezing)
INFO:train:Freeze policy applied: cnn: NONE (no freezing)
[2026-04-10 10:52:51] Freeze policy applied: enc_rnn: NONE (no freezing)
INFO:train:Freeze policy applied: enc_rnn: NONE (no freezing)
[2026-04-10 10:52:51] Freeze policy applied: attention: NONE (no freezing)
INFO:train:Freeze policy applied: attention: NONE (no freezing)
[2026-04-10 10:52:51] Parameters: trainable=10,618,692 | frozen=0 | total=10,618,692
INFO:train:Parameters: trainable=10,618,692 | frozen=0 | total=10,618,692
/usr/local/lib/python3.12/dist-packages/manuscript/recognizers/_trba/training/train.py:1103: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)`

...

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4445: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


[OK] ONNX model saved to: exp1/best_acc_model.onnx

Verifying ONNX model...
[OK] ONNX model is valid

Testing ONNX inference...
[OK] Exported ONNX model inference works!
  Output shape: (1, 26, 194)
  Max difference vs PyTorch: 0.000038
  [OK] Outputs match!
  [OK] Dynamic batch inference works for batch_size=2

Simplifying ONNX model...
[OK] ONNX model simplified

Validating simplified ONNX model...
[OK] Simplified ONNX model inference works!
  Output shape: (1, 26, 194)
  Max difference vs PyTorch: 0.000038
  [OK] Outputs match!


[2026-04-10 11:01:12] ONNX model exported successfully: exp1/best_acc_model.onnx
INFO:train:ONNX model exported successfully: exp1/best_acc_model.onnx


  [OK] Dynamic batch inference works for batch_size=2

[OK] Export complete! Model size: 40.5 MB

Input shape: [batch_size, 3, 64, 256]
Output shape: [batch_size, 26, 194]
Decoding: Greedy (argmax over last dimension)


{'val_acc': 0.8295535714285714,
 'val_loss': 0.16495008069356637,
 'exp_dir': 'exp1'}

По окончании обучения лучшая по точности модель автоматически конвертируется в `ONNX` формат.

Сообщение `Testing ONNX inference...` показывает, что после экспорта была выполнена проверка работоспособности ONNX-модели.

Строки

- `[OK] ONNX inference works!`
- `Output shape: (1, 26, 194)`
- `Max difference vs PyTorch: 0.000038`
- `[OK] Outputs match!`
- `[OK] Export complete!`

означают, что экспорт завершен успешно, ONNX-модель корректно выполняет инференс, а ее выходы практически совпадают с выходами исходной PyTorch-модели. Это показывает, что конвертация выполнена корректно и без существенных отклонений.

## 4. Инференс экспортированной модели

После обучения и экспорта модель распознавания можно сразу использовать для инференса. В данном примере загружается тестовый фрагмент `crop1.png` из основного репозитория и выполняется распознавание с помощью экспортированной модели `TRBA`.

In [5]:
from pathlib import Path
from urllib.request import urlretrieve

from manuscript.recognizers import TRBA
from manuscript.utils import create_page_from_image

exp_dir = Path("exp1")

image_url = "https://raw.githubusercontent.com/konstantinkozhin/manuscript-ocr/main/example/images/crop1.png"
image_path = Path("crop1.png")

if not image_path.exists():
    urlretrieve(image_url, image_path)

page = create_page_from_image(str(image_path))

recognizer = TRBA(
    weights=str(exp_dir / "best_acc_model.onnx"),
    config=str(exp_dir / "config.json"),
    charset=str(exp_dir / "charset.txt"),
)

result_page = recognizer.predict(page, image=str(image_path))

span = result_page.blocks[0].lines[0].text_spans[0]

print("Результат распознавания:")
print(result_page)
print("Текст:", span.text)
print("Confidence:", span.recognition_confidence)

[TRBA] Device configuration:
  Requested device: cpu
  Requested providers: ['CPUExecutionProvider']
  Active providers: ['CPUExecutionProvider']
  Running on: CPUExecutionProvider
Результат распознавания:
blocks=[Block(lines=[Line(text_spans=[TextSpan(polygon=[(0.0, 0.0), (456.0, 0.0), (456.0, 58.0), (0.0, 58.0)], detection_confidence=1.0, text='Никитинъ', recognition_confidence=0.9806256294250488, order=0)], order=0)], text_spans=[], order=0)]
Текст: Никитинъ
Confidence: 0.9806256294250488


Следует учитывать, что приведенный выше результат носит демонстрационный характер, поскольку модель в данном примере обучалась только `1` эпоху.

Таким образом, библиотека `manuscript-ocr` предоставляет возможность обучать распознаватели на собственных данных. Более подробная информация о параметрах обучения и доступных настройках приведена в полной документации проекта: https://konstantinkozhin.github.io/manuscript-ocr/0.1.12/en/api/recognizers.html